In [3]:
bundle_path: "EOInput" = "test_packaging3"
selection = "ensemble_simple"
input_dataset_path: "EOInput" = "input/elbe_bunthaus"
mode = "from_date"
start_date: str
# output_path: "EOInput" = "outputs/predictions.csv"

xcengine_config = dict(
  # ...
    include_directory=True,
    build_includes=["required_dist/"]  # paths relative to notebook
)

In [1]:
from pathlib import Path
from typing import get_type_hints

from pftnc.inference_config import InferenceConfig
from pftnc.predict.inference import run_inference

In [ ]:
output_path = "predictions.csv"

In [2]:
mode_type = get_type_hints(InferenceConfig)["mode"]
model_type = get_type_hints(InferenceConfig)["model"]
selection_type = get_type_hints(model_type)["selection"]

In [ ]:
selection: selection_type = selection
mode: mode_type = mode

In [4]:
import pystac

In [5]:
def extract_assets_from_catalog(catalog: pystac.Catalog, asset_key: str) -> list[pystac.Asset]:
    """
    Returns all assets with a given key from the items of a catalog.
    """
    assets = []
    for item in catalog.get_all_items():
        if (asset := item.assets.get(asset_key)) is not None:
            assets.append(asset)

    return assets

def get_catalog(inp: Path | str) -> pystac.Catalog:
    p = Path(inp) / "catalog.json"
    catalog = pystac.Catalog.from_file(p)
    catalog.make_all_asset_hrefs_absolute()
    return catalog

In [6]:
catalog_elbe_bunthaus = get_catalog(input_dataset_path)

In [7]:
catalog_elbe_bunthaus

<Catalog id=pftnc_elbe_bunthaus_catalog>

In [8]:
asset_id  = "elbe_bunthaus"
fpath_elbe_bunthaus = next(iter(extract_assets_from_catalog(catalog_elbe_bunthaus, asset_id))).href
fpath_elbe_bunthaus

'/home/yogesh/Projects/BC/ESA-Heatwise/heatwise-pftnc/eoap/input/elbe_bunthaus/inference_input.csv'

In [9]:
config = InferenceConfig(
      model={
          "bundle_path": Path(bundle_path),
          "selection": selection,
      },
      input_dataset={
          "path": Path(fpath_elbe_bunthaus),
      },
      mode=mode,
      start_date="2023-12-01",
      output_path=Path(output_path),
  )

In [10]:
predictions = run_inference(config)
print(predictions)

Generating rolling features:   0%|          | 0/360 [00:00<?, ?it/s]

                   time                      site       date   site_lat  \
223 2023-12-21 12:00:00  elbe-river_bunthaus_v0.3 2023-12-21  53.461787   
224 2023-12-22 12:00:00  elbe-river_bunthaus_v0.3 2023-12-22  53.461787   
225 2023-12-23 12:00:00  elbe-river_bunthaus_v0.3 2023-12-23  53.461787   
226 2023-12-24 12:00:00  elbe-river_bunthaus_v0.3 2023-12-24  53.461787   
227 2023-12-25 12:00:00  elbe-river_bunthaus_v0.3 2023-12-25  53.461787   
228 2023-12-27 12:00:00  elbe-river_bunthaus_v0.3 2023-12-27  53.461787   
229 2023-12-28 12:00:00  elbe-river_bunthaus_v0.3 2023-12-28  53.461787   
230 2023-12-29 12:00:00  elbe-river_bunthaus_v0.3 2023-12-29  53.461787   
231 2023-12-31 12:00:00  elbe-river_bunthaus_v0.3 2023-12-31  53.461787   

      site_lon  chime_diatoms_ug_per_l  chime_cyanobacteria_ug_per_l  \
223  10.064451                1.089561                      0.693202   
224  10.064451                1.380000                      1.100000   
225  10.064451                1.6

In [11]:
type(predictions)

pandas.core.frame.DataFrame